# Ablation Study: Neural PF Controller vs. Bootstrap PF

Compares the full model (neural PF controller, `use_net=True`) against a standard
bootstrap particle filter baseline (`use_net=False`) across all four paper datasets:
DS01, DS03, DS05, DS06.

Metrics reported: RMSE, NASA Score, PICP, PINAW — matching the paper's results table.

## Setup and Dependencies

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from experiment_config import (
    N_REP,
    SEED,
    UNCERTAINTY_LEVEL,
    DegModel,
    dataset_paths,
    pfnet_paths,
)
from src.helpers.metrics import build_metrics_table
from src.helpers.seed import set_global_seed
from src.models.networks.pf_mlp import ParticleFilterMLP, build_activation
from src.models.particle_filter.core import ParticleFilter
from src.models.rul_predictor import RULPredictor
from src.training.pfnet_hparams import PFNET_ARGS

## Parameters

In [ ]:
DATASETS = ["DS01", "DS03", "DS05", "DS06"]
VARIANTS = {"full": True, "baseline": False}

# ARGS_ID=0 is the dedicated ablation config; outputs are written under its dir.
# TRAINED_ARGS_ID is the trained model whose weights the "full" variant loads.
ARGS_ID = 0
TRAINED_ARGS_ID = 5
args = PFNET_ARGS[ARGS_ID]
net_args = args["NETWORK"]
pf_args = args["PARTICLE_FILTER"]

# Combined ablation outputs are written under the first dataset's arg dir.
_, PRED_DIR = pfnet_paths(ARGS_ID, DATASETS[0])


def build_paths(data_name: str):
    estimation_dir, degr_dir = dataset_paths(
        data_name, fields=["estimation", "degr_model"]
    )
    # "full" loads trained weights from TRAINED_ARGS_ID; outputs go under ARGS_ID
    pfnet_dir = degr_dir / f"net_arg{TRAINED_ARGS_ID}"
    pred_dir = degr_dir / f"net_arg{ARGS_ID}" / f"pred_ulevel{UNCERTAINTY_LEVEL}"
    return estimation_dir, degr_dir, pfnet_dir, pred_dir


set_global_seed(SEED)

## Run Ablation Across Datasets and Variants

In [ ]:
all_preds = []  # list of DataFrames

for data_name in DATASETS:
    estimation_dir, degr_dir, pfnet_dir, pred_dir = build_paths(data_name)
    pred_dir.mkdir(parents=True, exist_ok=True)

    # --- load HI data ---
    hi_df = pd.read_csv(estimation_dir / "data_dev.csv")
    dev_units = hi_df["unit"].astype(int).unique().tolist()
    perform_names = [c for c in hi_df.columns if c not in ["unit", "cycle", "hs"]]
    del hi_df

    hi_test = pd.read_csv(estimation_dir / "data_test.csv")
    test_units = hi_test["unit"].astype(int).unique().tolist()
    performs = {
        name: {u: hi_test[hi_test["unit"] == u][name].values for u in test_units}
        for name in perform_names
    }
    time = {u: hi_test[hi_test["unit"] == u]["cycle"].values for u in test_units}
    onsets = {
        u: hi_test[(hi_test["unit"] == u) & (hi_test["hs"] == 0)]["cycle"].values[0]
        for u in test_units
    }
    del hi_test

    # --- prepare test tensors ---
    test_data = {
        name: {
            u: torch.tensor(
                np.stack([time[u], performs[name][u]], axis=1), dtype=torch.float32
            )
            for u in test_units
        }
        for name in perform_names
    }

    for variant_name, use_net in VARIANTS.items():
        print(f"\n[{data_name}] variant={variant_name}")

        set_global_seed(SEED)

        # build pf models for this variant
        pfs = {}
        for perform_name in perform_names:
            net = ParticleFilterMLP(
                state_dim=DegModel.state_dim(),
                hidden_dims=list(net_args["HIDDEN_DIMS"]),
                activation=build_activation(net_args),
                dropout_p=float(net_args["DROPOUT"]),
            )
            if use_net:
                ckpt = torch.load(
                    pfnet_dir / perform_name / "checkpoint_best.pt",
                    weights_only=False,
                )
                net.load_state_dict(ckpt["model_state"])
            # baseline: untrained weights are ignored because use_net=False
            net = net.eval()

            degmodels = []
            for unit in dev_units:
                m = DegModel()
                m.load_state_dict(
                    torch.load(
                        degr_dir
                        / "states"
                        / perform_name
                        / f"unit_{unit}"
                        / "best_model.pt"
                    )
                )
                degmodels.append(m)

            pfs[perform_name] = ParticleFilter(
                base_models=degmodels,
                net=net,
                n_particles=int(pf_args["N_PARTICLES"]),
                max_life=int(pf_args["MAX_LIFE"]),
                use_net=use_net,
            )

        rul_pred = RULPredictor(
            pf_models=pfs,
            conf_level=UNCERTAINTY_LEVEL,
            max_life=int(pf_args["MAX_LIFE"]),
        )

        for rep in range(N_REP):
            print(f"  rep {rep + 1}/{N_REP}")
            rul_pred.reset()
            for test_unit in test_units:
                s_data = {name: performs[name][test_unit] for name in perform_names}
                t_data = time[test_unit]
                eol_time = t_data[-1]

                rul_pred.predict(t_data=t_data, s_data=s_data)

                df = rul_pred.history_to_dataframe()
                df["rep"] = rep
                df["unit"] = test_unit
                df["dataset"] = data_name
                df["variant"] = variant_name
                df["true_rul"] = np.maximum(eol_time - df["time"], 0.0)
                all_preds.append(df)


preds_df = pd.concat(all_preds, ignore_index=True)
print("\nDone. Saved to", PRED_DIR / "rul_ablation_all.csv")
preds_df.to_csv(PRED_DIR / "rul_ablation_all.csv", index=False)

## Metrics Table

In [ ]:
preds_df = pd.read_csv(PRED_DIR / "rul_ablation_all.csv")

# Average across repetitions, then compute metrics
mean_df = preds_df.groupby(
    ["dataset", "variant", "unit", "time"], as_index=False
).mean()

rows = []
for data_name in DATASETS:
    for variant_name in VARIANTS:
        df = mean_df[
            (mean_df["dataset"] == data_name) & (mean_df["variant"] == variant_name)
        ]
        m = build_metrics_table(df)
        rows.append({"Dataset": data_name, "Variant": variant_name, **m})

metrics_table = pd.DataFrame(rows).set_index(["Dataset", "Variant"])
metrics_table = metrics_table.round(4)
metrics_table

In [ ]:
metrics_table.to_csv(PRED_DIR / "ablation_metrics.csv")
print(metrics_table.to_string())

## RUL Plots — Full vs. Baseline (DS01)

In [ ]:
import matplotlib.pyplot as plt
from src.helpers.visualization import plot_rul_from_dataframe

ds01_df = mean_df[mean_df["dataset"] == "DS01"]
plot_units = ds01_df["unit"].unique().tolist()

colors = {"full": "steelblue", "baseline": "darkorange"}
_, pred_dir_ds01 = build_paths("DS01")[2], build_paths("DS01")[3]
pred_dir_ds01.mkdir(parents=True, exist_ok=True)

for unit in plot_units:
    fig, ax = plt.subplots(figsize=(8, 5))
    eol_time = None
    for variant_name in VARIANTS:
        df = ds01_df[(ds01_df["unit"] == unit) & (ds01_df["variant"] == variant_name)]
        eol_time = df["time"].iloc[-1]
        plot_rul_from_dataframe(
            ax=ax,
            df=df,
            t_max=eol_time,
            title=f"Ablation: DS01 unit {unit}",
            show_legend=(variant_name == "full"),
        )
        ax.lines[-1].set_label(variant_name)
        ax.lines[-1].set_color(colors[variant_name])

    ax.legend()
    fig.tight_layout()
    fig_path = pred_dir_ds01 / f"ablation_ds01_unit{unit}.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved {fig_path}")